# 3.3 — Data cleaning with an audit trail

Preserve the source, detect quality problems, and create analysis-ready data with documented rules, counts, actions, and validation.

## Introduction

Use this Notebook to verify the lesson concepts with actual data and code.

## Learning outcomes

- Keep source, working, and analysis-ready data separate.
- Detect conversion failures, missingness, label variation, invalid ranges, cross-field contradictions, and duplicate keys.
- Build a verification table that preserves every applicable issue reason.
- Explain the production of analysis-ready data through reconciled counts and an audit record.

> **Learning route:** Required: 3.3.1–3.3.6 / Integrated practice: 3.3.7


## 3.3.1 Preserve the source and define quality problems

Lesson 3.2 selected rows. Selection cannot produce reliable evidence if missing values, inconsistent labels, or impossible relationships remain unexplained. Preserve the source, profile problems, define rules, flag affected rows, choose an action, and validate counts and constraints. That order makes cleaning auditable.

In [ ]:
from pathlib import Path
import pandas as pd


def find_course_data(filename):
    """Find a course data file without depending on the notebook start folder."""
    roots = [Path.cwd(), *Path.cwd().parents, Path.home() / "work", Path("/opt/python-lab/course-materials")]
    checked = []
    for root in roots:
        for candidate in (root / "data" / filename, root / filename):
            candidate = candidate.expanduser()
            if candidate in checked:
                continue
            checked.append(candidate)
            if candidate.is_file():
                return candidate
    locations = "\n".join(f"- {candidate}" for candidate in checked)
    raise FileNotFoundError(f"Course data file {filename!r} was not found. Checked:\n{locations}")


data_file = find_course_data("learning-centres-practice.csv")
raw = pd.read_csv(data_file)
print("Loading:", data_file.resolve())
print("Rows:", len(raw), "Columns:", len(raw.columns))


## 3.3.2 Separate source, working, and analysis data

Display missing counts, data types, category values, and numeric ranges first. Keep `raw` unchanged and begin processing with `clean = raw.copy(deep=True)`. Without separate source and working data, you cannot verify what changed.

In [ ]:
print(raw.dtypes)
print("Missing values:\n", raw.isna().sum())
print("District labels:", sorted(raw["district"].dropna().unique()))
print(raw[["registered", "attended", "completed", "material_cost"]].describe())

clean = raw.copy(deep=True)


## 3.3.3 Handle conversion failures and missing values

A numeric CSV column containing text may be read as text. `pd.to_numeric(..., errors="coerce")` converts invalid text to missing, but a source blank and a new conversion failure are different problems. Compare masks before and after conversion and record the count.

In [ ]:
numeric_columns = ["registered", "attended", "completed", "training_hours", "material_cost"]
conversion_failures = {}
for column in numeric_columns:
    before_missing = clean[column].isna()
    converted = pd.to_numeric(clean[column], errors="coerce")
    failed = converted.isna() & ~before_missing
    conversion_failures[column] = int(failed.sum())
    clean[column] = converted
print("Conversion failures:", conversion_failures)


### Do not confuse missingness with zero

Filling a blank attendance count with zero changes “not reported” into “nobody attended” and changes rates and averages. Do not impute without evidence. Flag it with `isna()` and document whether it is excluded from a calculation or quarantined for review.

In [ ]:
missing_attended = clean["attended"].isna()
print("Missing attended:", int(missing_attended.sum()))
print(clean.loc[missing_attended, ["month", "centre_id", "attended", "completed"]])


## 3.3.4 Normalise labels while retaining source values

Whitespace and case variations can split one district into several groups. Apply `strip()` and `title()` to a new working value and count changed rows while preserving `district_raw`. Never merge merely similar words unless an agreed rule or mapping confirms that they mean the same category.

In [ ]:
clean["district_raw"] = clean["district"]
clean["district"] = clean["district"].astype("string").str.strip().str.title()
changed_district = clean["district_raw"].astype("string") != clean["district"]
print("Changed district labels:", int(changed_district.sum()))
print(clean.loc[changed_district, ["district_raw", "district"]].drop_duplicates())


## 3.3.5 Test ranges, cross-field constraints, and duplicates

For this dataset, counts must be non-negative, attendance cannot exceed registration, and completion cannot exceed attendance. Separate Boolean masks explain how many rows violate each rule. Count missingness separately from invalidity.

In [ ]:
negative_count = (clean[["registered", "attended", "completed"]] < 0).any(axis=1)
attendance_over_registered = clean["attended"].notna() & (clean["attended"] > clean["registered"])
completion_over_attendance = clean["completed"].notna() & clean["attended"].notna() & (clean["completed"] > clean["attended"])
negative_cost = clean["material_cost"].notna() & (clean["material_cost"] < 0)
print("Negative learner counts:", int(negative_count.sum()))
print("Attendance above registration:", int(attendance_over_registered.sum()))
print("Completion above attendance:", int(completion_over_attendance.sum()))
print("Negative material cost:", int(negative_cost.sum()))
print(clean.loc[completion_over_attendance, ["month", "centre_id", "registered", "attended", "completed"]])


### Use a business key to flag every duplicate row

Exact duplicate rows are not the only possible double entry. Define one record as centre, month, and course, then use `duplicated(..., keep=False)` to display every row in a duplicate group. The same centre in another month is legitimate, so the key definition comes first.

In [ ]:
business_key = ["centre_id", "month", "course"]
duplicate_key = clean.duplicated(subset=business_key, keep=False)
print("Rows with duplicate business keys:", int(duplicate_key.sum()))
print(clean.loc[duplicate_key].sort_values(business_key))


## 3.3.6 Build verification and audit evidence

Do not delete a row immediately after detecting a problem. Correct it only from an authoritative source, normalise it only under an explicit rule, or mark it missing or pending review when evidence is insufficient. Here an impossible completion is not guessed; it is flagged as not analysis-ready. Both `raw` and `clean` retain their row counts.

In [ ]:
clean["analysis_ready"] = ~(
    missing_attended | negative_count | attendance_over_registered
    | completion_over_attendance | negative_cost | duplicate_key
)
analysis = clean.loc[clean["analysis_ready"]].copy()
print("Raw rows:", len(raw))
print("Retained clean rows:", len(clean))
print("Analysis-ready rows:", len(analysis))
print("Flagged rows:", int((~clean["analysis_ready"]).sum()))


### Preserve row-level reasons in the verification report

Rule counts alone do not tell a reviewer which source rows need attention. Keep every individual flag, join its reason in one documented order, and select the original identifying columns into a verification table. If one row breaks several rules, retain every matching reason rather than only the first. The analysis exclusion count and the reviewer hand-off then come from the same decisions.


In [ ]:
issue_rules = [
    (missing_attended, "missing attended"),
    (completion_over_attendance, "completion above attendance"),
    (negative_cost, "negative material cost"),
    (duplicate_key, "duplicate business key"),
]
clean["issue"] = ""
for mask, label in issue_rules:
    clean.loc[mask, "issue"] = clean.loc[mask, "issue"] + label + "; "
clean["issue"] = clean["issue"].str.rstrip("; ")

verification_columns = ["month", "centre_id", "course", "issue"]
records_to_verify = (
    clean.loc[~clean["analysis_ready"], verification_columns]
    .sort_values(["month", "centre_id", "course"])
    .reset_index(drop=True)
)
records_to_verify


### Record the audit and validate again

An audit log summarises the decisions performed by code. Record the issue, detection rule, affected count, action, and remaining unresolved count. A zero count still documents that a check was performed.

In [ ]:
audit = pd.DataFrame([
    {"issue": "missing attended", "rule": "attended is missing", "affected": int(missing_attended.sum()), "action": "exclude from rate analysis; request review"},
    {"issue": "district spelling", "rule": "strip whitespace and title case", "affected": int(changed_district.sum()), "action": "normalise; preserve district_raw"},
    {"issue": "completion above attendance", "rule": "completed <= attended", "affected": int(completion_over_attendance.sum()), "action": "flag; do not guess replacement"},
    {"issue": "duplicate business key", "rule": "unique centre_id + month + course", "affected": int(duplicate_key.sum()), "action": "review duplicate group"},
])
audit


### Reapply constraints and reconcile counts

Apply the same constraints after the action and confirm that source count equals analysis-ready count plus flagged count. `assert` stops the workflow where an expectation fails, making a new monthly data problem visible.

In [ ]:
assert len(raw) == len(clean)
assert len(clean) == int(clean["analysis_ready"].sum()) + int((~clean["analysis_ready"]).sum())
assert not (analysis["completed"] > analysis["attended"]).any()
assert not analysis.duplicated(subset=business_key).any()
print("Validation passed")


## 3.3.7 Integrated practice: apply the quality workflow to another table

Check the practice CSV for missing completion, attendance above registration, non-positive training hours, negative material cost, and duplicate business keys. Display each mask count. Do not guess corrections; create an audit table containing the rule, count, and proposed action, then add a reconciliation assertion.

In [ ]:
# Write the transfer solution here.


## Summary

- Kept the source unchanged while adding quality flags to a working copy.
- Applied named rules and retained multiple reasons on the same row.
- Separated analysis and verification records, then rechecked counts, constraints, and the audit trail.

## Next

The valid and review-required records are now separated with evidence. Lesson 3.4 aggregates the analysis-ready records into indicators and priorities.

**Estimated learning time:** about 3 hours
